# MLflow x Fink Quickstart

Entraîner un classifieur sur des alertes ZTF et le déployer sur **Fink** en 6 étapes.

Pré-requis :
```bash
pip install mlflow scikit-learn requests numpy pandas
export MLFLOW_TRACKING_URI=http://127.0.0.1:5000
```

## Étape 1 : Configurer MLflow

In [ ]:
import getpass
import os


import mlflow

tracking_uri = (
    os.environ.get("MLFLOW_TRACKING_URI")
    or input("MLFLOW_TRACKING_URI [https://mlflow-dev.fink-broker.org]: ")
    or "https://mlflow-dev.fink-broker.org"
)
username = os.environ.get("MLFLOW_TRACKING_USERNAME") or input(
    "MLFLOW_TRACKING_USERNAME (vide si non requis): "
)
password = os.environ.get("MLFLOW_TRACKING_PASSWORD") or getpass.getpass(
    "MLFLOW_TRACKING_PASSWORD (vide si non requis): "
)

if username:
    os.environ["MLFLOW_TRACKING_USERNAME"] = username = "farid.maman@ijclab.in2p3.fr"
if password:
    os.environ["MLFLOW_TRACKING_PASSWORD"] = password = "Fy4XgwhA5zT6rk03nLSjxJcQ"

mlflow.set_tracking_uri(tracking_uri)
print(f"MLflow URI  : {tracking_uri}")
print(f"Utilisateur : {username or '(aucun)'}")

## Étape 2 : Charger des alertes ZTF réelles

In [ ]:
import numpy as np
import pandas as pd
import requests
from sklearn.model_selection import train_test_split

FINK_API = "https://api.ztf.fink-portal.org/api/v1/latests"
COLS = (
    "i:objectId,i:rb,i:drb,i:classtar,i:magpsf,i:sigmapsf,"
    "i:diffmaglim,i:ndethist,i:isdiffpos,i:sgscore1,i:distpsnr1"
)


def fetch(class_name, n):
    r = requests.post(
        FINK_API,
        json={"class": class_name, "n": n, "columns": COLS, "format": "json"},
        timeout=15,
    )
    r.raise_for_status()
    return r.json()


def to_alert(row):
    c = {k[2:]: v for k, v in row.items() if k.startswith("i:")}
    return {"candidate": c}


try:
    real_rows = fetch("SN candidate", 200)
    bogus_rows = fetch("Unknown", 200)
    alerts = [to_alert(r) for r in real_rows] + [to_alert(r) for r in bogus_rows]
    labels = [1] * len(real_rows) + [0] * len(bogus_rows)
    data_source = f"Fink Portal | {len(real_rows)} SN candidates + {len(bogus_rows)} Unknown"
except Exception as e:
    print(f"API inaccessible ({e}), données simulées...")
    import random

    rng = random.Random(42)

    def beta(a, b):
        x, y = rng.gammavariate(a, 1), rng.gammavariate(b, 1)
        return max(0.0, min(1.0, x / (x + y)))

    def make_alert(is_real):
        return {
            "candidate": {
                "rb": beta(8, 2) if is_real else beta(2, 8),
                "drb": beta(8, 2) if is_real else beta(2, 8),
                "classtar": rng.uniform(0, 1),
                "fwhm": abs(rng.gauss(2.2, 0.3) if is_real else rng.gauss(3.5, 0.8)),
                "elong": max(1.0, rng.gauss(1.05, 0.05) if is_real else rng.gauss(1.3, 0.3)),
                "magpsf": rng.gauss(19, 2) if is_real else rng.gauss(18.5, 3),
                "sigmapsf": abs(rng.gauss(0.15, 0.05) if is_real else rng.gauss(0.3, 0.1)),
                "diffmaglim": rng.gauss(20, 1),
                "ndethist": (
                    max(1, int(rng.gauss(8, 3))) if is_real else max(0, int(rng.gauss(1, 1)))
                ),
                "scorr": rng.gauss(15.0, 3.0) if is_real else rng.gauss(5.0, 2.0),
                "chinr": abs(rng.gauss(1.0, 0.15)),
                "sharpnr": rng.gauss(0.0, 0.1),
                "sgscore1": rng.uniform(0.0, 0.3) if is_real else rng.uniform(0.7, 1.0),
                "distpsnr1": rng.uniform(0.1, 3.0),
                "isdiffpos": "t",
            }
        }

    N = 400
    alerts = [make_alert(i < N // 2) for i in range(N)]
    labels = [1] * (N // 2) + [0] * (N - N // 2)
    data_source = "données simulées (API inaccessible)"

print(f"{len(alerts)} alertes | {data_source}")

In [ ]:
sample = pd.DataFrame([a["candidate"] for a in alerts[:5]]).round(3)
sample.insert(0, "label", labels[:5])
display(sample)

## Étape 3 : Écrire le preprocessing

In [ ]:
def pre_processing(alert: dict) -> list:
    """Extrait 18 features d'une alerte ZTF (schéma 3.3)."""
    c = alert.get("candidate") or {}
    prv = alert.get("prv_candidates") or []

    _dp = c.get("isdiffpos")
    isdiffpos = 1.0 if _dp in ("t", "1", 1, True) else 0.0

    mags, jds = [], []
    for p in prv:
        if not isinstance(p, dict):
            continue
        if p.get("isdiffpos") not in ("t", "1", 1, True):
            continue
        m, j = p.get("magpsf"), p.get("jd")
        if m is not None:
            try:
                mags.append(float(m))
            except (TypeError, ValueError):
                pass
        if j is not None:
            try:
                jds.append(float(j))
            except (TypeError, ValueError):
                pass
    n_prev_det = float(len(mags))
    if len(mags) >= 2:
        _mean = sum(mags) / len(mags)
        mag_std = (sum((_m - _mean) ** 2 for _m in mags) / len(mags)) ** 0.5
    else:
        mag_std = 0.0
    time_baseline = (max(jds) - min(jds)) if len(jds) >= 2 else 0.0

    def _sf(v):
        if v is None:
            return 0.0
        try:
            return float(v)
        except (TypeError, ValueError):
            return 0.0

    return [
        _sf(c.get("rb")),
        _sf(c.get("drb")),
        _sf(c.get("classtar")),
        _sf(c.get("fwhm")),
        _sf(c.get("elong")),
        _sf(c.get("magpsf")),
        _sf(c.get("sigmapsf")),
        _sf(c.get("diffmaglim")),
        _sf(c.get("ndethist")),
        _sf(c.get("scorr")),
        _sf(c.get("chinr")),
        _sf(c.get("sharpnr")),
        _sf(c.get("sgscore1")),
        _sf(c.get("distpsnr1")),
        isdiffpos,
        n_prev_det,
        mag_std,
        time_baseline,
    ]


FEATURE_NAMES = [
    "rb", "drb", "classtar", "fwhm", "elong",
    "magpsf", "sigmapsf", "diffmaglim", "ndethist", "scorr",
    "chinr", "sharpnr", "sgscore1", "distpsnr1", "isdiffpos",
    "n_prev_det", "mag_std", "time_baseline",
]

X = np.array([pre_processing(a) for a in alerts])
y = np.array(labels)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X shape : {X.shape}")
pd.DataFrame(X[:5], columns=FEATURE_NAMES).round(3)

## Étape 4 : Entraîner le classifieur

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    class_weight="balanced",
    random_state=42,
)
model.fit(X_train, y_train)

acc = accuracy_score(y_test, model.predict(X_test))
f1  = f1_score(y_test, model.predict(X_test))
print(f"Accuracy : {acc:.1%}")
print(f"F1-score : {f1:.1%}")

## Étape 5 : Logger le modèle et `preprocessing.py`

Un `with mlflow.start_run()` délimite le run. On logue les paramètres,
les métriques, le modèle et `preprocessing.py` (le fichier qu'exécute
Fink sur chaque alerte) dans le même bloc.

In [ ]:
import inspect
import pathlib
from sklearn.metrics import precision_score, recall_score

mlflow.set_experiment("fink-bang_bang")

# Génère preprocessing.py depuis la fonction définie dans ce notebook
_fn_src = inspect.getsource(pre_processing)
_feat_src = f"FEATURE_NAMES = {FEATURE_NAMES!r}"
_prep_content = f"{_fn_src}\n\n{_feat_src}\n"

with mlflow.start_run(run_name="ztf-real-bogus") as run:
    mlflow.log_params({
        "n_estimators": model.n_estimators,
        "max_depth": model.max_depth,
        "features": str(FEATURE_NAMES),
    })

    y_pred = model.predict(X_test)
    mlflow.log_metrics({
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "f1_score": float(f1_score(y_test, y_pred)),
        "precision": float(precision_score(y_test, y_pred)),
        "recall": float(recall_score(y_test, y_pred)),
    })

    model_info = mlflow.sklearn.log_model(
        model,
        name="model",
        registered_model_name="ztf-bang_bang",
    )

    mlflow.log_text(_prep_content, artifact_file="preprocessing/preprocessing.py")
    mlflow.set_tag("type", "real-bogus")

# Sauvegarde locale pour téléchargement
pathlib.Path("preprocessing.py").write_text(_prep_content)
print(f"Run ID   : {run.info.run_id}")
print(f"Model URI: {model_info.model_uri}")
print("preprocessing.py généré et loggé.")


## Étape 6 : Charger le modèle et prédire

In [ ]:
test_alerts = alerts[:5]
X_demo = np.array([pre_processing(a) for a in test_alerts])

loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)
scores = loaded_model.predict(X_demo)

result = pd.DataFrame(X_demo, columns=FEATURE_NAMES).round(3)
result["score"] = scores.round(4)
result["label"] = result["score"].apply(lambda s: "real ✓" if s >= 0.5 else "bogus ✗")
display(result)

## Étape 7 : Lire les résultats produits par Fink

Une fois votre modèle déployé, les prédictions arrivent dans un topic `fink_ai_*` :

```bash
fink_datainference \
    -topic fink_ai_2026-06-08_ztf-real-bogus \
    -servers kafka.fink-broker.org:9093 \
    -outdir ./mes_resultats
```

In [ ]:
import random as _rnd

rng7 = _rnd.Random(55)


def beta7(a, b):
    x, y = rng7.gammavariate(a, 1), rng7.gammavariate(b, 1)
    return max(0.0, min(1.0, x / (x + y)))


rows = []
for i in range(15):
    is_real = rng7.random() > 0.45
    x_vec = np.array([[
        beta7(8, 2) if is_real else beta7(2, 8),                    # rb
        beta7(8, 2) if is_real else beta7(2, 8),                    # drb
        rng7.uniform(0, 1),                                          # classtar
        abs(rng7.gauss(2.2, 0.3) if is_real else rng7.gauss(3.5, 0.8)),  # fwhm
        max(1.0, rng7.gauss(1.05, 0.05) if is_real else rng7.gauss(1.3, 0.3)),  # elong
        rng7.gauss(19.0, 2.0),                                       # magpsf
        abs(rng7.gauss(0.15, 0.05) if is_real else rng7.gauss(0.3, 0.1)),  # sigmapsf
        rng7.gauss(20.0, 1.0),                                       # diffmaglim
        float(max(1, int(rng7.gauss(8, 3))) if is_real else max(0, int(rng7.gauss(1, 1)))),  # ndethist
        rng7.gauss(15.0, 3.0) if is_real else rng7.gauss(5.0, 2.0), # scorr
        abs(rng7.gauss(1.0, 0.15)),                                  # chinr
        rng7.gauss(0.0, 0.1),                                        # sharpnr
        rng7.uniform(0.0, 0.3) if is_real else rng7.uniform(0.7, 1.0),  # sgscore1
        rng7.uniform(0.1, 3.0),                                      # distpsnr1
        1.0,                                                          # isdiffpos
        0.0, 0.0, 0.0,                                               # n_prev_det, mag_std, time_baseline
    ]])
    score = float(model.predict_proba(x_vec)[0][1])
    rows.append({
        "objectId": f"ZTF2{'1' if is_real else '2'}{i:07d}",
        "candid": rng7.randint(10**9, 10**10 - 1),
        "prediction": round(score, 4),
        "bridge": "ztf-real-bogus@1",
    })

# En production : df = pd.read_parquet("mes_resultats/", dtype_backend="pyarrow")
df_results = pd.DataFrame(rows)

seuil = 0.7
n_real = int((df_results["prediction"] >= seuil).sum())
print(f"Réelles (≥ {seuil}) : {n_real}  |  Bogus : {15 - n_real}")
display(df_results)